# Task 2 — F-cls: Qwen2.5-Coder-7B + QLoRA + head 10 nhãn (Kaggle T4×2)

Fine-tune phân loại đa nhãn (xem `docs/task2_finetune_plan.md`). Model đọc đề + code + kết quả compile/chạy rồi cho xác suất của 10 nhãn.
Ngưỡng từng nhãn được chọn theo out-of-fold trên 32 bài gold, sau đó áp luật gold. Public chỉ dùng để dự đoán.

**Chuẩn bị**
1. Settings: Accelerator **GPU T4 x2**, Internet **On**.
2. Secrets (Add-ons → Secrets):
   - `PUBLIC_TEST_DRIVE_ID`: file id Drive của bộ dữ liệu cuộc thi.
   - `SILVER_DRIVE_ID`: file id Drive của **`generate-dataset_v3.zip`**.
   - Tuỳ chọn: `GITHUB_TOKEN`, `HF_TOKEN`.
   - Không dán id vào notebook rồi commit: repo là public.
3. `BRANCH = 'task-2-fcls'`.

**Chạy:** *Save Version → Save & Run All (Commit)*. Notebook train 3 run:
- `EX01` và `EX02` (2 fold, chạy song song trên 2 GPU) để đo out-of-fold trên gold;
- `all` (model final) để dự đoán public.

**Khi bị timeout / dừng giữa chừng.** Mỗi run lưu checkpoint (adapter + optimizer + scheduler) mỗi `save_steps` step vào `/kaggle/working/cls/<run>/checkpoints/`.
Ngoài ra script tự **dừng sạch trước giới hạn phiên** (`SESSION_HOURS`), để notebook kết thúc bình thường và Kaggle lưu `/kaggle/working`.
Run chưa xong thì `status.json` = `paused`. Để chạy tiếp:
1. Mở notebook → **Add Input** → *Your Work / Notebook Output* → chọn version vừa chạy của chính notebook này (được mount vào `/kaggle/input/...`).
2. *Save & Run All* lại. Script tìm checkpoint mới nhất trong `/kaggle/input` và resume đúng step đó. Run đã `done` thì bỏ qua.
- Tuỳ chọn chống crash cứng (mất điện, OOM): đặt `HUB_REPO = '<user>/task2-cls-ckpt'` + secret `HF_TOKEN`. Checkpoint cuối được đẩy lên HF Hub (repo private) sau mỗi lần lưu và được kéo về khi resume.

**Output** (`/kaggle/working`):
- `task2_predictions.json` (nộp bài);
- `cls/<name>_report/`: `report.md`, `metrics.json` (fixed / cross-fold tuned / in-sample), `oof_gold.json`;
- `cls/<run>/`: `train_log.jsonl`, `run_info.json` (phiên bản model, thư viện, GPU, VRAM đỉnh), `adapter/`.


In [ ]:
import time
T0 = time.time()               # session start: the training deadline is counted from here
REPO = 'github.com/tiendinhquang2104/llm-final-project.git'
BRANCH = 'task-2-fcls'
DRIVE_ID = ''                  # or secret PUBLIC_TEST_DRIVE_ID (never commit the id: the repo is public)
SILVER_DRIVE_ID = ''           # generate-dataset_v3.zip, or secret SILVER_DRIVE_ID
WORK = '/kaggle/tmp'           # repo + data (not saved with the version)
OUT = '/kaggle/working/cls'    # runs, checkpoints, reports (saved with the version -> resume next time)
CONFIG = 'configs/task2/cls_qwen7b.yaml'
SESSION_HOURS = 11.5           # commit sessions stop at 12h; interactive sessions: set what is left
HUB_REPO = ''                  # e.g. 'myuser/task2-cls-ckpt' (+ secret HF_TOKEN): mirror of the last checkpoint
TAG = ''                       # suffix of the run folders, e.g. 'lr2e-4' for a grid point
EPOCHS = None                  # override train.epochs (None = config)
LR = None                      # override train.lr
SMOKE = False                  # True: 16 training rows, 1 epoch (checks the whole pipeline in ~15 min)


In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ''
token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
os.makedirs(WORK, exist_ok=True)
REPO_DIR = f'{WORK}/repo'
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone failed — check REPO/BRANCH/GITHUB_TOKEN') from None  # never print the token URL
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)
os.chdir(REPO_DIR)
if secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret('HF_TOKEN')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q gdown 'transformers>=4.51,<5' 'peft>=0.13' 'bitsandbytes>=0.45' accelerate safetensors scikit-learn pyyaml
!g++ --version | head -1
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import gdown, shutil
drive_id = DRIVE_ID or secret('PUBLIC_TEST_DRIVE_ID')
assert drive_id, 'Set DRIVE_ID (or the PUBLIC_TEST_DRIVE_ID secret)'
if not os.path.exists('data/raw/sample_dataset'):
    os.makedirs(f'{WORK}/download', exist_ok=True)
    assert gdown.download(id=drive_id, output=f'{WORK}/download/data.zip', quiet=False), 'gdown failed (public data)'
    !python scripts/unpack_data.py {WORK}/download --dest data/raw
SYNTH_ZIP = 'data/synth/v1/generate-dataset_v3.zip'   # = data.synth_zip in the config
if not os.path.exists(SYNTH_ZIP):
    found = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs if f == 'generate-dataset_v3.zip']
    os.makedirs(os.path.dirname(SYNTH_ZIP), exist_ok=True)
    if found:
        shutil.copy(found[0], SYNTH_ZIP)
    else:
        sid = SILVER_DRIVE_ID or secret('SILVER_DRIVE_ID')
        assert sid, 'Set SILVER_DRIVE_ID (generate-dataset_v3.zip) or add the zip as a Kaggle input'
        assert gdown.download(id=sid, output=SYNTH_ZIP, quiet=False), 'gdown failed (generate-dataset_v3.zip)'
!ls -la data/raw {SYNTH_ZIP}


In [ ]:
# Resume: bring back the run folders of a previous version of this notebook (Add Input -> Notebook Output).
# Finished runs (status done) are not trained again; paused runs continue from their last checkpoint.
import glob, json, shutil
os.makedirs(OUT, exist_ok=True)
for st in glob.glob('/kaggle/input/**/cls/*/status.json', recursive=True):
    src = os.path.dirname(st)
    dst = f'{OUT}/{os.path.basename(src)}'
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
        print('restored', os.path.basename(src), json.load(open(st)).get('state'))


In [ ]:
# Records: compile + run on the statement examples (g++), gold mutants, leakage guard. Deterministic, a few minutes.
import json
!python scripts/build_task2_cls_data.py --config {CONFIG}
summary = json.load(open('data/sft/task2_cls/summary.json'))
os.makedirs(OUT, exist_ok=True)
shutil.copy('data/sft/task2_cls/summary.json', f'{OUT}/data_summary.json')
print(json.dumps({k: summary[k] for k in ('n', 'by_source', 'folds', 'empty', 'compiler')}, ensure_ascii=False, indent=1))
assert summary['compiler'], 'no compiler: compile_log would be missing (train/serve mismatch)'


In [ ]:
# Train: the 2 folds in parallel (one GPU each), then the final model. Every run stops cleanly before DEADLINE.
import sys
DEADLINE = T0 + SESSION_HOURS * 3600
def cmd(holdout):
    c = [sys.executable, 'scripts/train_task2_cls.py', '--config', CONFIG, '--holdout', holdout, '--out-root', OUT,
         '--deadline', str(DEADLINE), '--resume-dir', '/kaggle/input']
    if HUB_REPO: c += ['--hub-repo', HUB_REPO]
    if TAG: c += ['--tag', TAG]
    if EPOCHS: c += ['--epochs', str(EPOCHS)]
    if LR: c += ['--lr', str(LR)]
    if SMOKE: c += ['--limit-train', '16', '--epochs', '1']
    return c
def launch(holdout, gpu):
    log = open(f'{OUT}/{holdout}{"_" + TAG if TAG else ""}.log', 'a')
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu), 'PYTHONUNBUFFERED': '1',
           'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}  # less fragmentation (OOM fix)
    return subprocess.Popen(cmd(holdout), stdout=log, stderr=subprocess.STDOUT, env=env)
def tail(holdout, n=5):
    p = f'{OUT}/{holdout}{"_" + TAG if TAG else ""}.log'
    print(f'--- {holdout}:', *open(p).read().splitlines()[-n:], sep='\n  ')
import torch
n_gpu = torch.cuda.device_count()
codes = {}
if n_gpu >= 2:
    procs = {'EX01': launch('EX01', 0), 'EX02': launch('EX02', 1)}
    while any(p.poll() is None for p in procs.values()):
        time.sleep(300)
        for h in procs: tail(h, 2)
    codes.update({h: p.returncode for h, p in procs.items()})
else:
    for h in ('EX01', 'EX02'):
        codes[h] = launch(h, 0).wait()
codes['all'] = launch('all', 0).wait()
for h in codes: tail(h)
print('exit codes (0 = done, 3 = paused by the deadline):', codes)
PAUSED = [h for h, c in codes.items() if c == 3]
FAILED = [h for h, c in codes.items() if c not in (0, 3)]
if PAUSED:
    print(f'PAUSED {PAUSED}: Add Input -> this notebook\'s latest version output, then Save & Run All again (resumes).')
assert not FAILED, f'runs failed: {FAILED} — see {OUT}/<holdout>.log'


In [ ]:
# Out-of-fold scores on gold, thresholds, public predictions (only when every run is done)
if not PAUSED:
    !python scripts/eval_task2_cls.py --config {CONFIG} --out-root {OUT} {('--tag ' + TAG) if TAG else ''}
    import glob
    rep = glob.glob(f'{OUT}/*_report{"_" + TAG if TAG else ""}')[0]
    if os.path.exists(f'{rep}/predictions.json'):
        shutil.copy(f'{rep}/predictions.json', '/kaggle/working/task2_predictions.json')
        print('-> /kaggle/working/task2_predictions.json')
    from IPython.display import Markdown, display
    display(Markdown(open(f'{rep}/report.md').read()))


In [ ]:
# Space used by checkpoints (keep /kaggle/working under 20 GB)
!du -sh {OUT}/* 2>/dev/null | sort -h | tail
